In [7]:
from mirror import MirrorCode
import time

# Getting started with mirror codes
**Before you begin, you will need to install `stim`!**

In this short notebook tutorial, we will show how to generate a mirror code on a group $G$ and subsets $A, B$ of $G$.
For simplicity, we will focus on the case when the group is abelian.

The fundamental theorem of abelian groups says that every abelian group is just $\mathbb{Z}_{q_1} \times \cdots \times \mathbb{Z}_{q_m}$, where each $q_i$ is an integer. (In fact, you can take them to be prime powers.)
Therefore, we will specify an abelian group $G$ by giving a tuple of integers $(q_1, \dots, q_m)$.
Each element of the group will be of the form $(z_1, \dots, z_m)$, where $0 \leq z_i < q_i$.
We will then specify the subsets $A, B$ each as a list of these elements.

Consider, for example, the group $G = \mathbb{Z}_{2} \times \mathbb{Z}_{2} \times \mathbb{Z}_{3} \times \mathbb{Z}_{3}$.
The check weight of a mirror code is $|A| + |B|$, so let's set each subset to size 3 to give weight 6 checks.
Consider, for example, $A = [(0, 0, 0, 0), (0, 1, 0, 1), (1, 0, 0, 2)]$ and $B = [(0,0,0,0), (0,1,1,0), (1,1,2,0)]$.
Let's make this code! We'll also print its stabilizers. 

Note that a mirror code always has $n = |G|$ physical qubits and the same number of stabilizers (some are not linearly independent from the rest).

In [15]:
G = [2, 2, 3, 3]
A = [[0, 0, 0, 0],
     [0, 1, 0, 1],
     [1, 0, 0, 2]]
B = [[0, 0, 0, 0],
     [0, 1, 1, 0],
     [1, 1, 2, 0]]

code = MirrorCode(group=G, z0=A, x0=B)

print(f"n = {code.get_n()}, k = {code.get_k()}\n")

# Print the code's stabilizers in symplectic representation. This is a 2n x n binary matrix.
# Each row is a stabilizer, where the first (last) n bits encode whether or not the qubit has a Z (X).
print("=== Stabilizers in symplectic form ===")
print(code.get_stabilizers())
print("\n")

# You might also want the stabilizers directly in stim stabilizer tableau format.
print("=== Stabilizers in stim format ===")
stim_stabs = code.get_stim_tableau() 
for stab in stim_stabs:
    print(stab)

n = 36, k = 6

=== Stabilizers in symplectic form ===
[[1 0 0 ... 1 0 0]
 [0 1 0 ... 0 0 1]
 [0 0 1 ... 0 1 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


=== Stabilizers in stim format ===
+Y_________Z_X_______Z____________X__
+_ZX________Z__X___Z________________X
+_XZ______Z___X_____Z______________X_
+___Z__X__X___Z_________Z______X_____
+____Z___X__X__Z______Z__________X___
+_____Z_X__X_Z_________Z________X____
+___X__Z________XZ_________ZX________
+_____X_Z_________Y______Z____X______
+____X___Z______ZX________Z__X_______
+_Z_X_____Y______________X____Z______
+__Z__X____ZX______________XZ________
+Z___X_____XZ_____________X__Z_______
+X___Z_______Z__X_____X__________Z___
+__X__Z_______Z___X_____X______Z_____
+_X_Z__________Z_X_____X________Z____
+______XZ____X__Z__X________________Z
+________Y_____X_Z___X____________Z__
+______ZX_____X___Z_X______________Z_
+__Z____________X__Y_________Z_X_____
+Z________________X_ZX________Z__X___
+_Z______________X__XZ______Z

In [14]:
# What is the distance of the code we just generated?
# We can use stim's distance calculator to find out.
d = code.get_d()
print(f"Distance = {d}")

Distance = 6


**Note**: if you replace this example with something that has a much higher distance, computing distance could take a while.
For example, let's try a code whose parameters you might [recognize](https://arxiv.org/abs/2308.07915v2):

In [ ]:
G = [2, 6, 12]
A = [[0, 0, 0],
     [0, 0, 1],
     [0, 3, 2]]
B = [[1, 0, 0],
     [1, 1, 0],
     [1, 2, 3]]
code2 = MirrorCode(group=G, z0=A, x0=B)
print(f"n = {code2.get_n()}, k = {code2.get_k()}")

# This takes about 3 minutes on a decent laptop
start = time.time()
d = code2.get_d()
end = time.time()

print(f"Distance = {d}")
print(f"Took {end - start:0.2f} s")

n = 144, k = 12
Distance = 12
Took 176.93 s


In [11]:
# If it takes too long, you can use a distance *estimator* instead of an exact calculator, in which case you get
# an upper bound which hopefully isn't too far off.

start = time.time()
d_est = code2.get_d_est()
end = time.time()

print(f"Estimated distance = {d_est}")
print(f"Took {end - start:0.5f} s")

Estimated distance = 12
Took 0.00002 s


Lastly, 

## Non-abelian mirror codes
Mirror codes can also be defined on non-abelian groups, but the choice of subsets $A, B$ are then restricted in such a way that ensures the stabilizers produced actually commute. 
Also, since the order of multiplication matters, there turns out to be two natural formulations of non-abelian mirror codes which are generally not equivalent. 
One has stabilizers of the form $\boldsymbol{Z}(Ag) \boldsymbol{X}(B g^{-1})$ for each $g \in G$; we call these *symmetric mirror codes*. 
The other has stabilizers of the form $\boldsymbol{Z}(Ag) \boldsymbol{X}(g^{-1} B)$; we call these *asymmetric mirror codes*.

The simplest way to choose $A, B$ to ensure commutation is just to put them in the center $Z(G)$ of the group. In this case, the symmetric and asymmetric codes are identical.
It turns out that the most general condition for commutation can be written as, in the (a)symmetric case: the quantity $|Agh \cap B| = |Ahg \cap B|$ for all $g, h \in G$ ($|gAh \cap B| = |hAg \cap B|$ for all $g, h \in G$).